#  **1 - Importação, Filtros e Pré-processamento**

Nesta etapa, importamos os dados da ONU (World Population Prospects 2024). 
Aplicaremos dois filtros iniciais de qualidade recomendados pela literatura:
1. Manter apenas países individuais, removendo agregados regionais (utilizando a presença do `ISO3_code`).
2. Excluir países com populações abaixo de 1 milhão de habitantes no ano recente para evitar instabilidade estatística nas taxas de probabilidade de morte.

### **1.1 - Configuração do Ambiente**

Importação das bibliotecas base para manipulação dos dados estruturados e gerenciamento dos caminhos de arquivo.

In [1]:
import pandas as pd
from pathlib import Path

### **1.2 - Leitura dos Dados**

Para evitar falhas de leitura (`ParserError`), utilizamos uma abordagem de importação blindada. O código detecta automaticamente o delimitador do arquivo, lida com caracteres especiais (usando `utf-8-sig`) e converte strings nulas ou vazias corretamente. Em seguida, selecionamos apenas os indicadores de mortalidade e expectativa de vida pertinentes à modelagem.

In [2]:
file_path = "../data/WPP2024_Demographic_Indicators_Medium.csv" # Ajuste o caminho se necessário

# 1. Leitura blindada inspirada no _common.py para evitar ParserError
with open(file_path, "r", encoding="utf-8-sig") as f:
    first_line = f.readline()

# Detecta o separador correto
sep = ";" if first_line.count(";") > first_line.count(",") else ","

In [3]:
# Carrega o CSV usando os parâmetros seguros
df_raw = pd.read_csv(
    file_path, 
    sep=sep, 
    encoding="utf-8-sig", 
    low_memory=False,
    keep_default_na=False, 
    na_values=["", "NaN", "nan", "NULL", "null"]
)

# 2. Seleção das colunas de interesse para a nossa modelagem
cols = [
    'Location', 'ISO3_code', 'Time', 'TPopulation1July', 
    'LEx', 'Q5', 'Q1560', 'LE15', 'LE65', 'LE80'
]
df = df_raw[cols].copy()

# Forçar conversão numérica nas colunas de dados
numeric_cols = ['TPopulation1July', 'LEx', 'Q5', 'Q1560', 'LE15', 'LE65', 'LE80']
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

### **1.3 - Filtragem de Qualidade da Base**

De acordo com a literatura demográfica ([Araújo](https://arxiv.org/abs/2609.17268) & [Aburto](https://doi.org/10.1073/pnas.1915884117)), aplicamos dois filtros críticos:
- **Isolamento de Países Reais:** Removemos regiões econômicas e geográficas agregadas exigindo que a coluna `ISO3_code` esteja preenchida.
- **Estabilidade Estatística:** Excluímos países com população inferior a 1 milhão (baseado no ano de 2023) aplicado conforme [Araújo (2026)](https://doi.org/10.1073/pnas.1915884117), utilizando o último ano de estimativas históricas para evitar viés de pequenos países insulares nas trajetórias de mortalidade

In [4]:
# 3. Filtro 1: Apenas países (ISO3_code preenchido)
df = df[df['ISO3_code'].notna() & (df['ISO3_code'] != "")]

# 4. Filtro 2: Excluir países com população < 1 milhão no ano base (ex: 2023)
pop_2023 = df[df['Time'] == 2023]
paises_validos = pop_2023[pop_2023['TPopulation1July'] >= 1000]['ISO3_code'].unique()

df_filtered = df[df['ISO3_code'].isin(paises_validos)].copy()

print(f"Total de países na base bruta (ISO3 identificados): {df['ISO3_code'].nunique()}")
print(f"Total de países após filtro de população (>1M): {df_filtered['ISO3_code'].nunique()}")

Total de países na base bruta (ISO3 identificados): 237
Total de países após filtro de população (>1M): 161


In [5]:
df_filtered.head()

,Location,ISO3_code,Time,TPopulation1July,LEx,Q5,Q1560,LE15,LE65,LE80
44384,Burundi,BDI,1950,2254.938,40.9382,276.1295,423.9696,44.9163,11.0477,4.7283
44385,Burundi,BDI,1951,2305.746,41.2292,273.1188,421.0379,45.0586,11.1012,4.7577
44386,Burundi,BDI,1952,2355.804,41.4802,270.3369,418.7733,45.1673,11.1395,4.7776
44387,Burundi,BDI,1953,2405.186,41.7429,267.4953,416.3086,45.2853,11.1807,4.7995
44388,Burundi,BDI,1954,2454.586,41.9731,264.8267,414.4213,45.3754,11.2109,4.8142


In [6]:
df_filtered.info()

<class 'pandas.DataFrame'>
Index: 24472 entries, 44384 to 81319
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Location          24472 non-null  str    
 1   ISO3_code         24472 non-null  str    
 2   Time              24472 non-null  int64  
 3   TPopulation1July  24311 non-null  float64
 4   LEx               24311 non-null  float64
 5   Q5                24311 non-null  float64
 6   Q1560             24311 non-null  float64
 7   LE15              24311 non-null  float64
 8   LE65              24311 non-null  float64
 9   LE80              24311 non-null  float64
dtypes: float64(7), int64(1), str(2)
memory usage: 2.1 MB


### **1.4 - Inspeção e Tratamento de Valores Nulos**

A modelagem de séries temporais requer continuidade. Vamos verificar se existem países com dados faltantes (`NaN`) nas variáveis selecionadas. A célula abaixo demonstra o que aconteceria se aplicássemos uma remoção de nulos sem analisar o contexto temporal da base.

In [7]:
# 5. Tratamento de Dados Nulos (Missing Values)
# Identifica quais países possuem QUALQUER valor nulo nas colunas numéricas
paises_com_nulos = df_filtered[df_filtered[numeric_cols].isna().any(axis=1)]['Location'].unique()

print(f"Países com dados faltantes que serão removidos: {list(paises_com_nulos)}")

# Filtra o dataframe mantendo apenas os países que NÃO estão na lista de nulos
df_clean = df_filtered[~df_filtered['Location'].isin(paises_com_nulos)].copy()

print("-" * 50)
print(f"Total de países iniciais: {df_filtered['ISO3_code'].nunique()}")
print(f"Total de países final para a modelagem: {df_clean['ISO3_code'].nunique()}")

Países com dados faltantes que serão removidos: ['Burundi', 'Djibouti', 'Eritrea', 'Ethiopia', 'Kenya', 'Madagascar', 'Malawi', 'Mauritius', 'Mozambique', 'Rwanda', 'Somalia', 'South Sudan', 'Uganda', 'United Republic of Tanzania', 'Zambia', 'Zimbabwe', 'Angola', 'Cameroon', 'Central African Republic', 'Chad', 'Congo', 'Democratic Republic of the Congo', 'Equatorial Guinea', 'Gabon', 'Algeria', 'Egypt', 'Libya', 'Morocco', 'Sudan', 'Tunisia', 'Botswana', 'Eswatini', 'Lesotho', 'Namibia', 'South Africa', 'Benin', 'Burkina Faso', "Côte d'Ivoire", 'Gambia', 'Ghana', 'Guinea', 'Guinea-Bissau', 'Liberia', 'Mali', 'Mauritania', 'Niger', 'Nigeria', 'Senegal', 'Sierra Leone', 'Togo', 'Kazakhstan', 'Kyrgyzstan', 'Tajikistan', 'Turkmenistan', 'Uzbekistan', 'China', 'China, Hong Kong SAR', 'China, Taiwan Province of China', "Dem. People's Republic of Korea", 'Japan', 'Mongolia', 'Republic of Korea', 'Afghanistan', 'Bangladesh', 'India', 'Iran (Islamic Republic of)', 'Nepal', 'Pakistan', 'Sri Lank

### **1.5 Correção do Horizonte Temporal**

**Descoberta Crítica:** O bloco anterior eliminaria todos os países válidos. Isso acontece porque a base WPP 2024 projeta a população total até o ano de 2101, mas as taxas de mortalidade detalhadas encerram no ano de 2100, gerando uma linha final com valores nulos para todos os países.

Para preservar os dados, restringimos o horizonte temporal de análise até o ano 2100. Depois dessa restrição, garantimos a limpeza final removendo eventuais resíduos.

In [8]:
# 5. Tratamento de Dados Nulos (Missing Values) - CORRIGIDO
# A base vai até 2101, mas as taxas de mortalidade param em 2100.
# Restringimos o tempo para evitar o descarte de países válidos.

# Filtro de tempo: manter apenas até 2100
df_clean = df_filtered[df_filtered['Time'] <= 2100].copy()

# Removemos qualquer resíduo de nulo que possa ter sobrado nas variáveis de mortalidade
df_clean = df_clean.dropna(subset=numeric_cols)

print("-" * 50)
print(f"Total de países inicial: 161")
print(f"Total de países final para a modelagem: {df_clean['ISO3_code'].nunique()}")
print(f"Período de análise: {df_clean['Time'].min()} até {df_clean['Time'].max()}")

--------------------------------------------------
Total de países inicial: 161
Total de países final para a modelagem: 161
Período de análise: 1950 até 2100


### **1.6 - Padronização da Nomenclatura das Variáveis**

As siglas originais da ONU (como `LEx`, `Q5`, `Q1560`) são jargões demográficos que dificultam a leitura do código por terceiros. Para melhorar a legibilidade (Data Storytelling), vamos renomear as colunas para termos descritivos em português.

In [9]:
# Dicionário de tradução das colunas
dicionario_nomes = {
    'Location': 'Pais',
    'ISO3_code': 'Codigo_ISO3',
    'Time': 'Ano',
    'TPopulation1July': 'Populacao_Total',
    'LEx': 'Exp_Vida_Nascer',
    'Q5': 'Mort_Infantil_Sub5',
    'Q1560': 'Mort_Adulta_15_60',
    'LE15': 'Exp_Vida_15',
    'LE65': 'Exp_Vida_65',
    'LE80': 'Exp_Vida_80'
}

# Aplicando a renomeação ao dataframe limpo
df_clean = df_clean.rename(columns=dicionario_nomes)

# Verificando como ficaram os novos nomes
df_clean.info()

<class 'pandas.DataFrame'>
Index: 24311 entries, 44384 to 81318
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Pais                24311 non-null  str    
 1   Codigo_ISO3         24311 non-null  str    
 2   Ano                 24311 non-null  int64  
 3   Populacao_Total     24311 non-null  float64
 4   Exp_Vida_Nascer     24311 non-null  float64
 5   Mort_Infantil_Sub5  24311 non-null  float64
 6   Mort_Adulta_15_60   24311 non-null  float64
 7   Exp_Vida_15         24311 non-null  float64
 8   Exp_Vida_65         24311 non-null  float64
 9   Exp_Vida_80         24311 non-null  float64
dtypes: float64(7), int64(1), str(2)
memory usage: 2.0 MB


Vamos salvar essa novo *dataset* em uma pasta de dados exlcusiva para os dados gerados a partir dos *notebooks*:

In [10]:
import os

# Define o caminho da pasta e do arquivo
output_dir = '../new_data'
file_path = os.path.join(output_dir, 'wpp2024_clean.csv')

# Cria a pasta se ela não existir; se já existir, não faz nada
os.makedirs(output_dir, exist_ok=True)

# Salva o dataset limpo
df_clean.to_csv(file_path, index=False)